In [1]:
#Import Libs
import os.path
import xarray as xr
import numpy as np

In [2]:
#Functions that will be used for postprocessing
class apce_data_struct:
    def __init__(self, t, ds_t, int_OD, RHi):
        self.t = t
        self.ds_t = ds_t
        self.int_OD = int_OD
        self.RHi = RHi
    
def read_apcemm_data(directory):
    t_mins = []
    ds_t = []
    int_OD = []
    RHi = []

    for file in sorted(os.listdir(directory)):
        if(file.startswith('ts_aerosol') and file.endswith('.nc')):
            file_path = os.path.join(directory,file)
            ds = xr.open_dataset(file_path, engine = "netcdf4", decode_times = False)
            ds_t.append(ds)
            tokens = file_path.split('.')
            mins = int(tokens[-2][-2:])
            hrs = int(tokens[-2][-4:-2])
            t_mins.append(hrs*60 + mins)
            int_OD.append(ds["intOD"])
            RHi.append(ds["RHi"])

    return apce_data_struct(t_mins, ds_t, int_OD, RHi)

In [3]:
# Extract integrated vertical optical depth data from APCEMM output files
training_sample_matrix_output = []
training_sample_matrix_input = []
training_sample_arrays = []
scaled_mean = 120 # Reducing back to [0,1] space after APCEMM scaling
scaling_factor = 15 # Reducing back to [0,1] space after APCEMM scaling

for i in range(1, 11): # For test runs 1-10
    apce_data = read_apcemm_data('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/outputs/test_1_run_' + str(i))
    int_OD = apce_data.int_OD
    current_training_sample_output = np.array(int_OD).reshape(1, 73)[0]
    training_sample_matrix_output.append(current_training_sample_output)
    
    # Define your input_RHi array (length 24)
    input_RHi_ds = xr.open_dataset('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_1/APCEMM_met_run_'+ str(i) +'.nc')
    input_RHi = input_RHi_ds['relative_humidity_ice'][98].values
    
    # Expand input_RHi to match the timestamps
    expanded_input_RHi = [] 
    expanded_input_RHi.append(input_RHi[0]) # Count the zeroth timestep as a 7th repeat
    repeated_input_RHi = np.repeat(input_RHi[0:12], 6)
    expanded_input_RHi.extend(repeated_input_RHi) # 1 hour is just 6 10-minute intervals, and APCEMM is only run for 12 hours
    normalized_input_RHi = (np.array(expanded_input_RHi) - scaled_mean) / scaling_factor
    current_training_sample_input = np.array(normalized_input_RHi).reshape(1, 73)[0]
    training_sample_matrix_input.append(current_training_sample_input)
    
    # Stacking the input and output arrays
    stacked = np.dstack((current_training_sample_input, current_training_sample_output))
    training_sample_arrays.append(stacked)
    input_RHi_ds.close()

training_sample_matrix = np.vstack(training_sample_arrays) # Shape: (10, 73, 2) INPUTS: (:,:,0), OUTPUTS: (:,:,1)

1
2
3
4
5
6
7
8
9
10


In [4]:
# Save the training_sample_matrix to a .npy file
np.save('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_1/training_sample_matrix.npy', training_sample_matrix)

In [5]:
loaded_training_sample_matrix = np.load('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_1/training_sample_matrix.npy')
print(loaded_training_sample_matrix[:,:,0])

[[-1.52666667e-01 -1.52666667e-01 -1.52666667e-01 -1.52666667e-01
  -1.52666667e-01 -1.52666667e-01 -1.52666667e-01  4.59333333e-01
   4.59333333e-01  4.59333333e-01  4.59333333e-01  4.59333333e-01
   4.59333333e-01 -8.51333333e-01 -8.51333333e-01 -8.51333333e-01
  -8.51333333e-01 -8.51333333e-01 -8.51333333e-01  4.24000000e-01
   4.24000000e-01  4.24000000e-01  4.24000000e-01  4.24000000e-01
   4.24000000e-01 -4.04666667e-01 -4.04666667e-01 -4.04666667e-01
  -4.04666667e-01 -4.04666667e-01 -4.04666667e-01 -1.21266667e+00
  -1.21266667e+00 -1.21266667e+00 -1.21266667e+00 -1.21266667e+00
  -1.21266667e+00 -1.16800000e+00 -1.16800000e+00 -1.16800000e+00
  -1.16800000e+00 -1.16800000e+00 -1.16800000e+00 -4.73333333e-01
  -4.73333333e-01 -4.73333333e-01 -4.73333333e-01 -4.73333333e-01
  -4.73333333e-01 -2.72000000e-01 -2.72000000e-01 -2.72000000e-01
  -2.72000000e-01 -2.72000000e-01 -2.72000000e-01 -4.04666667e-01
  -4.04666667e-01 -4.04666667e-01 -4.04666667e-01 -4.04666667e-01
  -4.04666